# Cell2Cell Cluster Transfer

Train clusters only on Cell2Cell and test transfer to IBM Telco and BankChurners.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler


SCRIPT_DIR = Path(__file__).resolve().parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from abstract_shared_ibm_test import (  # noqa: E402
    ROOT,
    RANDOM_STATE,
    build_abstract_bank,
    build_abstract_cell,
    build_abstract_ibm,
    best_f1_threshold,
    compute_metrics,
    fit_imputer,
    fit_xgb,
    load_bank,
    load_cell2cell,
    load_ibm,
    md_table,
    split_domain,
    transform_imputer,
)


RESULT_PATH = ROOT / "results" / "cell2cell_cluster_transfer_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "cell2cell_cluster_transfer.ipynb"


def fit_clusterer(X_train: pd.DataFrame, k: int):
    imp, X_train_imp = fit_imputer(X_train)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train_imp)
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    kmeans.fit(X_scaled)
    train_cluster = pd.Series(kmeans.labels_, index=X_train_imp.index, name="cluster")
    return {
        "imputer": imp,
        "train_imp": X_train_imp,
        "scaler": scaler,
        "kmeans": kmeans,
        "train_cluster": train_cluster,
    }


def predict_cluster(clusterer: dict, X: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X_imp = transform_imputer(clusterer["imputer"], X).reset_index(drop=True)
    X_scaled = clusterer["scaler"].transform(X_imp)
    cluster = pd.Series(clusterer["kmeans"].predict(X_scaled), index=X_imp.index, name="cluster")
    return X_imp, cluster


def add_cluster_one_hot(X_imp: pd.DataFrame, cluster: pd.Series, k: int) -> pd.DataFrame:
    cluster = cluster.to_numpy()
    cluster_df = pd.DataFrame({f"cluster_{i}": (cluster == i).astype(float) for i in range(k)}, index=X_imp.index)
    return pd.concat([X_imp, cluster_df], axis=1)


def fit_router(X_train: pd.DataFrame, y_train: pd.Series, cluster_train: pd.Series, min_cluster_size: int = 250):
    global_model = fit_xgb(X_train, y_train)
    models: dict[int, object] = {}
    for c in sorted(cluster_train.unique()):
        mask = cluster_train == c
        y_c = y_train.loc[mask]
        if int(mask.sum()) < min_cluster_size or y_c.nunique() < 2:
            continue
        models[int(c)] = fit_xgb(X_train.loc[mask], y_c)
    return {"global": global_model, "models": models}


def router_predict(router: dict, X: pd.DataFrame, cluster: pd.Series) -> np.ndarray:
    prob = np.zeros(len(X), dtype=float)
    cluster_np = cluster.to_numpy()
    for c in np.unique(cluster_np):
        mask = cluster_np == c
        model = router["models"].get(int(c), router["global"])
        prob[mask] = model.predict_proba(X.iloc[mask])[:, 1]
    return prob


def cluster_quality(cluster: pd.Series, y: pd.Series, X_scaled: np.ndarray) -> dict:
    cluster_np = cluster.to_numpy()
    y_np = y.to_numpy()
    counts = pd.Series(cluster_np).value_counts().sort_index()
    churn_table = pd.crosstab(cluster_np, y_np)
    return {
        "n_clusters": int(len(counts)),
        "cluster_min_size": int(counts.min()),
        "cluster_max_size": int(counts.max()),
        "churn_ari": float(adjusted_rand_score(y_np, cluster_np)),
        "churn_nmi": float(normalized_mutual_info_score(y_np, cluster_np)),
        "churn_purity": float(churn_table.max(axis=1).sum() / churn_table.to_numpy().sum()),
        "silhouette": float(silhouette_score(X_scaled, cluster_np, sample_size=min(5000, len(cluster_np)), random_state=RANDOM_STATE)),
    }


def summarize_target(name: str, baseline_prob: np.ndarray, aug_prob: np.ndarray, router_prob: np.ndarray, y_true: pd.Series, target_best: bool = True):
    rows = [
        {"setting": f"{name} baseline", **compute_metrics(y_true, baseline_prob)},
        {"setting": f"{name} cluster-aug", **compute_metrics(y_true, aug_prob)},
        {"setting": f"{name} cluster-router", **compute_metrics(y_true, router_prob)},
    ]
    if target_best:
        for row, prob in zip(rows, [baseline_prob, aug_prob, router_prob]):
            t, f1, metrics = best_f1_threshold(y_true, prob)
            row["best_threshold"] = t
            row["best_f1"] = f1
            row["best_auc"] = metrics["roc_auc"]
    return pd.DataFrame(rows)


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# Cell2Cell Cluster Transfer\n",
                    "\n",
                    "Train clusters only on Cell2Cell and test transfer to IBM Telco and BankChurners.\n",
                ],
            },
            {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": script_text.splitlines(keepends=True)},
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bank(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")
    ibm_df = load_ibm(ROOT / "data" / "external" / "ibm_telco" / "Telco-Customer-Churn.csv")

    cell_X, cell_y = build_abstract_cell(cell_df)
    bank_X, bank_y = build_abstract_bank(bank_df)
    ibm_X, ibm_y = build_abstract_ibm(ibm_df)

    cX_train, cX_test, cy_train, cy_test = split_domain(cell_X, cell_y)
    bX_train, bX_test, by_train, by_test = split_domain(bank_X, bank_y)
    iX_train, iX_test, iy_train, iy_test = split_domain(ibm_X, ibm_y)

    # Cell2Cell-only source model baseline.
    cell_imp, cX_train_imp = fit_imputer(cX_train)
    cX_test_imp = transform_imputer(cell_imp, cX_test)
    bank_test_imp = transform_imputer(cell_imp, bX_test)
    ibm_test_imp = transform_imputer(cell_imp, iX_test)
    baseline_model = fit_xgb(cX_train_imp, cy_train)

    baseline_table = pd.DataFrame(
        [
            {"setting": "Cell2Cell holdout", **compute_metrics(cy_test, baseline_model.predict_proba(cX_test_imp)[:, 1])},
            {"setting": "BankChurners holdout", **compute_metrics(by_test, baseline_model.predict_proba(bank_test_imp)[:, 1])},
            {"setting": "IBM holdout", **compute_metrics(iy_test, baseline_model.predict_proba(ibm_test_imp)[:, 1])},
        ]
    )

    k_values = list(range(2, 9))
    sweep_rows = []
    best_by_external = None
    best_score = -np.inf
    per_k_tables = []

    for k in k_values:
        clusterer = fit_clusterer(cX_train, k)
        train_cluster = clusterer["train_cluster"]
        X_scaled_train = clusterer["scaler"].transform(clusterer["train_imp"])

        c_test_imp, c_test_cluster = predict_cluster(clusterer, cX_test)
        bank_test_imp_k, bank_test_cluster = predict_cluster(clusterer, bX_test)
        ibm_test_imp_k, ibm_test_cluster = predict_cluster(clusterer, iX_test)

        train_aug = add_cluster_one_hot(clusterer["train_imp"], train_cluster, k)
        c_test_aug = add_cluster_one_hot(c_test_imp, c_test_cluster, k)
        bank_test_aug = add_cluster_one_hot(bank_test_imp_k, bank_test_cluster, k)
        ibm_test_aug = add_cluster_one_hot(ibm_test_imp_k, ibm_test_cluster, k)

        aug_model = fit_xgb(train_aug, cy_train)
        router = fit_router(clusterer["train_imp"], cy_train, train_cluster)
        router_c_prob = router_predict(router, c_test_imp, c_test_cluster)
        router_b_prob = router_predict(router, bank_test_imp_k, bank_test_cluster)
        router_i_prob = router_predict(router, ibm_test_imp_k, ibm_test_cluster)

        baseline_c_prob = baseline_model.predict_proba(cX_test_imp)[:, 1]
        baseline_b_prob = baseline_model.predict_proba(bank_test_imp)[:, 1]
        baseline_i_prob = baseline_model.predict_proba(ibm_test_imp)[:, 1]
        aug_c_prob = aug_model.predict_proba(c_test_aug)[:, 1]
        aug_b_prob = aug_model.predict_proba(bank_test_aug)[:, 1]
        aug_i_prob = aug_model.predict_proba(ibm_test_aug)[:, 1]

        sweep_rows.append(
            {
                "k": k,
                **cluster_quality(train_cluster, cy_train, X_scaled_train),
                "cell_baseline_auc": compute_metrics(cy_test, baseline_c_prob)["roc_auc"],
                "bank_baseline_auc": compute_metrics(by_test, baseline_b_prob)["roc_auc"],
                "ibm_baseline_auc": compute_metrics(iy_test, baseline_i_prob)["roc_auc"],
                "cell_aug_auc": compute_metrics(cy_test, aug_c_prob)["roc_auc"],
                "bank_aug_auc": compute_metrics(by_test, aug_b_prob)["roc_auc"],
                "ibm_aug_auc": compute_metrics(iy_test, aug_i_prob)["roc_auc"],
                "cell_router_auc": compute_metrics(cy_test, router_c_prob)["roc_auc"],
                "bank_router_auc": compute_metrics(by_test, router_b_prob)["roc_auc"],
                "ibm_router_auc": compute_metrics(iy_test, router_i_prob)["roc_auc"],
            }
        )
        per_k_tables.append(
            {
                "k": k,
                "cell": summarize_target("Cell2Cell", baseline_c_prob, aug_c_prob, router_c_prob, cy_test),
                "bank": summarize_target("BankChurners", baseline_b_prob, aug_b_prob, router_b_prob, by_test),
                "ibm": summarize_target("IBM", baseline_i_prob, aug_i_prob, router_i_prob, iy_test),
            }
        )

        score = (
            compute_metrics(by_test, router_b_prob)["roc_auc"]
            + compute_metrics(iy_test, router_i_prob)["roc_auc"]
        ) / 2.0
        if score > best_score:
            best_score = score
            best_by_external = {
                "k": k,
                "cell": summarize_target("Cell2Cell", baseline_c_prob, aug_c_prob, router_c_prob, cy_test),
                "bank": summarize_target("BankChurners", baseline_b_prob, aug_b_prob, router_b_prob, by_test),
                "ibm": summarize_target("IBM", baseline_i_prob, aug_i_prob, router_i_prob, iy_test),
            }

    sweep_df = pd.DataFrame(sweep_rows)

    result_lines = []
    result_lines.append("# Cell2Cell-only Cluster Transfer")
    result_lines.append("")
    result_lines.append("## Goal")
    result_lines.append("- Train clusters only on Cell2Cell.")
    result_lines.append("- Test the learned cluster structure on BankChurners and IBM Telco separately.")
    result_lines.append("- Compare a plain baseline, a cluster-augmented model, and a cluster-router model.")
    result_lines.append("")
    result_lines.append("## Cell2Cell baseline transfer")
    result_lines.append(md_table(baseline_table.round(4)))
    result_lines.append("")
    result_lines.append("## K sweep")
    result_lines.append(md_table(sweep_df.round(4)))
    result_lines.append("")
    result_lines.append(f"## Best external-average router k={best_by_external['k']}")
    result_lines.append(best_by_external["cell"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append(best_by_external["bank"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append(best_by_external["ibm"].round(4).to_string(index=False))
    result_lines.append("")
    result_lines.append("## Interpretation")
    result_lines.append("- If the router improves IBM more than BankChurners, then the Cell2Cell cluster structure is telecom-specific.")
    result_lines.append("- If BankChurners remains weak, then Cell2Cell clusters do not transfer cleanly into the financial domain.")
    result_lines.append("- The best k is selected by the average external AUC on IBM and BankChurners.")
    RESULT_PATH.write_text("\n".join(result_lines), encoding="utf-8")

    print("Wrote:", RESULT_PATH)
    print()
    print("=== Cell2Cell baseline transfer ===")
    print(md_table(baseline_table.round(4)))
    print()
    print("=== K sweep ===")
    print(md_table(sweep_df.round(4)))
    print()
    print(f"=== Best external-average router k={best_by_external['k']} ===")
    print(best_by_external["cell"].round(4).to_string(index=False))
    print()
    print(best_by_external["bank"].round(4).to_string(index=False))
    print()
    print(best_by_external["ibm"].round(4).to_string(index=False))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
